# NFL Draft Combine Predictor
## GCI World 2026 April · In-Class Competition

**Author:** Omokhoa Oshose Tosayoname (Tosa) · `26UNN_Tosayoname`  
**Task:** Predict the probability that an NFL Combine athlete will be selected in the NFL Draft.  
**Metric:** ROC-AUC · **Best Public Score:** 0.82983

| Step | Detail |
|---|---|
| Imputation | Position-wise group mean (7 columns) + missing-value flags |
| Encoding | Smoothed school target encoding (k=10) + group draft-rate encodings |
| Features | 9 position-normalised z-scores + 6 physical composites = 33 total |
| Model | LightGBM + XGBoost · 5-Fold Stratified CV · Optuna-tuned (100 trials) |
| Ensemble | 60% LightGBM + 40% XGBoost, averaged across 5 folds |

**Contents:**  
1. Setup and Data Loading  
2. Exploratory Data Analysis  
3. Feature Engineering  
4. Model Training and Cross-Validation  
5. Feature Importance  
6. Generate Submission File  


## 1. Setup and Data Loading

In [ ]:
# Uncomment on first run in Colab
# !pip install lightgbm xgboost optuna -q

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    'figure.facecolor':'#0a0f1e','axes.facecolor':'#111827','axes.edgecolor':'#1e2d45',
    'text.color':'#e2e8f0','axes.labelcolor':'#94a3b8','xtick.color':'#94a3b8',
    'ytick.color':'#94a3b8','grid.color':'#1e2d45','grid.linestyle':'--','grid.alpha':0.5,
})
GOLD='#f0b429'; GREEN='#22c55e'; RED='#ef4444'; BLUE='#3b82f6'
print(f'LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | Optuna {optuna.__version__}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IMPORTANT: Change this to your actual competition folder path
%cd "/content/drive/MyDrive/2026 GCI_Global/competition"

from pathlib import Path
PATH = Path.cwd() / 'input'
assert (PATH / 'train.csv').exists(), 'train.csv not found. Check your PATH.'
assert (PATH / 'test.csv').exists(),  'test.csv not found. Check your PATH.'
print('All files found.')

train_raw  = pd.read_csv(PATH / 'train.csv')
test_raw   = pd.read_csv(PATH / 'test.csv')
sample_sub = pd.read_csv(PATH / 'sample_submission.csv')

print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')
print(f'Drafted: {int((train_raw["Drafted"]==1).sum())} ({train_raw["Drafted"].mean():.1%}) | Not Drafted: {int((train_raw["Drafted"]==0).sum())} ({1-train_raw["Drafted"].mean():.1%})')

## 2. Exploratory Data Analysis

In [ ]:
# --- Missing value rates ---
missing_cols = ['Age','Sprint_40yd','Vertical_Jump','Bench_Press_Reps','Broad_Jump','Agility_3cone','Shuttle']
miss_rate = train_raw[missing_cols].isnull().mean().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(11, 4))
colors = [RED if v > 0.25 else GOLD if v > 0.10 else GREEN for v in miss_rate.values]
bars = ax.barh(miss_rate.index, miss_rate.values*100, color=colors, height=0.6, alpha=0.85)
for bar, val in zip(bars, miss_rate.values):
    ax.text(val*100+0.5, bar.get_y()+bar.get_height()/2, f'{val:.1%}', va='center', ha='left', fontsize=10, color='#e2e8f0', fontweight='600')
ax.set_xlabel('Missing Rate (%)', fontsize=11)
ax.set_title('Missing Value Rates by Feature', fontsize=13, color='#e2e8f0', pad=14, fontweight='600')
ax.set_xlim(0, 45); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()
print('Note: Age_missing ranks #3 in feature importance — absence is itself predictive.')

In [ ]:
# --- Draft rate by position ---
pos_stats = train_raw.groupby('Position')['Drafted'].agg(['mean','count']).reset_index()
pos_stats.columns = ['Position','DraftRate','Count']
pos_stats = pos_stats.sort_values('DraftRate', ascending=True)
fig, ax = plt.subplots(figsize=(11, 6))
colors = [GREEN if v >= 0.65 else GOLD if v >= 0.50 else RED for v in pos_stats['DraftRate']]
bars = ax.barh(pos_stats['Position'], pos_stats['DraftRate']*100, color=colors, height=0.65, alpha=0.85)
for bar, count, rate in zip(bars, pos_stats['Count'], pos_stats['DraftRate']):
    ax.text(rate*100+0.5, bar.get_y()+bar.get_height()/2, f'{rate:.1%}  (n={count})', va='center', ha='left', fontsize=9, color='#e2e8f0')
ax.axvline(65, color=GOLD, linestyle='--', alpha=0.4, linewidth=1)
ax.set_xlabel('Draft Rate (%)', fontsize=11)
ax.set_title('NFL Draft Selection Rate by Position (2009-2019)', fontsize=13, color='#e2e8f0', pad=14, fontweight='600')
ax.set_xlim(0, 95); ax.grid(axis='x', alpha=0.3)
legend_elements = [mpatches.Patch(color=GREEN, label='High (>=65%)'), mpatches.Patch(color=GOLD, label='Mid (50-65%)'), mpatches.Patch(color=RED, label='Low (<50%)')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, facecolor='#111827', edgecolor='#1e2d45')
plt.tight_layout(); plt.show()

In [ ]:
# --- Combine metrics: drafted vs not drafted ---
perf_cols = ['Sprint_40yd','Vertical_Jump','Bench_Press_Reps','Broad_Jump','Agility_3cone','Shuttle']
drafted = train_raw[train_raw['Drafted']==1]
not_drafted = train_raw[train_raw['Drafted']==0]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Combine Metrics Distribution: Drafted vs. Not Drafted', fontsize=13, color='#e2e8f0', fontweight='600', y=1.01)
for ax, col in zip(axes.flat, perf_cols):
    ax.hist(drafted[col].dropna(), bins=30, alpha=0.7, color=GREEN, label='Drafted', density=True)
    ax.hist(not_drafted[col].dropna(), bins=30, alpha=0.7, color=RED, label='Not Drafted', density=True)
    ax.set_title(col.replace('_',' '), fontsize=10, color='#e2e8f0', pad=6)
    ax.set_ylabel('Density', fontsize=8); ax.grid(alpha=0.2)
    ax.legend(fontsize=8, facecolor='#111827', edgecolor='#1e2d45')
plt.tight_layout(); plt.show()

In [ ]:
# --- Top schools by draft rate ---
school_stats = (train_raw.groupby('School')['Drafted'].agg(['mean','count']).query('count >= 15').sort_values('mean', ascending=True).tail(15))
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(school_stats.index, school_stats['mean']*100, color=GOLD, height=0.65, alpha=0.8)
for bar, (idx, row) in zip(bars, school_stats.iterrows()):
    ax.text(row['mean']*100+0.3, bar.get_y()+bar.get_height()/2, f"{row['mean']:.1%}  (n={int(row['count'])})", va='center', ha='left', fontsize=9, color='#e2e8f0')
ax.axvline(train_raw['Drafted'].mean()*100, color='#4b6080', linestyle='--', linewidth=1, label=f"Overall mean ({train_raw['Drafted'].mean():.1%})")
ax.set_xlabel('Draft Rate (%)', fontsize=11)
ax.set_title('Top 15 Schools by Draft Rate (min. 15 attendees)', fontsize=13, color='#e2e8f0', pad=14, fontweight='600')
ax.set_xlim(0, 95); ax.grid(axis='x', alpha=0.3)
ax.legend(fontsize=9, facecolor='#111827', edgecolor='#1e2d45')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

In [ ]:
def engineer_features(df_train, df_test):
    """
    Full feature engineering pipeline.
    All statistics computed exclusively from df_train. No data leakage.
    Returns: (engineered_train_df, engineered_test_df)
    """
    train = df_train.copy(); test = df_test.copy()
    MISSING_COLS = ['Age','Sprint_40yd','Vertical_Jump','Bench_Press_Reps','Broad_Jump','Agility_3cone','Shuttle']
    PERF_COLS    = ['Sprint_40yd','Vertical_Jump','Bench_Press_Reps','Broad_Jump','Agility_3cone','Shuttle','Height','Weight','Age']
    DROP_COLS    = ['Id','School','Player_Type','Position_Type','Position']

    # Step 1: Missing value flags (before any imputation)
    for col in MISSING_COLS:
        train[col+'_missing'] = train[col].isnull().astype(int)
        test[col+'_missing']  = test[col].isnull().astype(int)

    # Step 2: Position-wise group mean imputation
    for col in MISSING_COLS:
        group_means = train.groupby('Position')[col].mean()
        global_mean = train[col].mean()
        fill = lambda r, gm=group_means, glb=global_mean, c=col: gm.get(r['Position'], glb) if pd.isnull(r[c]) else r[c]
        train[col] = train.apply(fill, axis=1)
        test[col]  = test.apply(fill, axis=1)

    # Step 3: Smoothed school target encoding (k=10)
    sc = train.groupby('School')['Drafted'].count()
    sm = train.groupby('School')['Drafted'].mean()
    gmd = train['Drafted'].mean(); k = 10
    se = (sc*sm + k*gmd) / (sc+k)
    train['School_encoded'] = train['School'].map(se).fillna(gmd)
    test['School_encoded']  = test['School'].map(se).fillna(gmd)

    # Step 4: Group draft-rate encodings
    for feat, grp in [('Position_rate','Position'),('PositionType_rate','Position_Type'),('PlayerType_rate','Player_Type')]:
        r = train.groupby(grp)['Drafted'].mean()
        train[feat] = train[grp].map(r); test[feat] = test[grp].map(r)

    # Step 5: Physical composite features
    for df in [train, test]:
        df['BMI']               = df['Weight'] / (df['Height']**2)
        df['Agility_composite'] = -df['Agility_3cone'] - df['Shuttle']
        df['Power_composite']   = df['Vertical_Jump'] + df['Broad_Jump']/10 + df['Bench_Press_Reps']
        df['Speed_power_ratio'] = df['Broad_Jump'] / (df['Sprint_40yd'] + 1e-6)
        df['Vert_Broad_ratio']  = df['Vertical_Jump'] / (df['Broad_Jump'] + 1e-6)
        df['Cone_Shuttle_diff'] = df['Agility_3cone'] - df['Shuttle']

    # Step 6: Position-normalised z-scores for all performance metrics
    for col in PERF_COLS:
        pm = train.groupby('Position')[col].mean()
        ps = train.groupby('Position')[col].std().replace(0, 1)
        zscore = lambda r, pm=pm, ps=ps, c=col: (r[c] - pm.get(r['Position'], train[c].mean())) / ps.get(r['Position'], train[c].std())
        train[col+'_pos_z'] = train.apply(zscore, axis=1)
        test[col+'_pos_z']  = test.apply(zscore, axis=1)

    train = train.drop(columns=DROP_COLS); test = test.drop(columns=DROP_COLS)
    return train, test

train_fe, test_fe = engineer_features(train_raw, test_raw)
X = train_fe.drop(columns=['Drafted']).copy()
y = train_fe['Drafted'].copy()
X_test = test_fe.copy()

# Fill any residual NaN with column median
for col in X.columns:
    med = X[col].median()
    X[col] = X[col].fillna(med); X_test[col] = X_test[col].fillna(med)

print(f'Features: {X.shape[1]} | NaN remaining: {X.isnull().sum().sum()}')

## 4. Model Training and Cross-Validation

In [ ]:
# Optuna-tuned LightGBM hyperparameters (100 trials)
lgb_params = {
    'objective':'binary','metric':'auc','n_estimators':2000,
    'learning_rate':0.05542674329174862,'num_leaves':100,'max_depth':3,
    'min_child_samples':16,'subsample':0.7548500625444321,
    'colsample_bytree':0.7094181071509991,'reg_alpha':0.003604904077584873,
    'reg_lambda':0.038553801878934285,'random_state':42,'verbose':-1,
}
xgb_params = {
    'objective':'binary:logistic','eval_metric':'auc','n_estimators':1000,
    'learning_rate':0.05,'max_depth':4,'subsample':0.8,'colsample_bytree':0.8,
    'reg_alpha':0.1,'reg_lambda':1.0,'early_stopping_rounds':100,
    'random_state':42,'verbosity':0,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_cv_scores=[]; xgb_cv_scores=[]; ens_cv_scores=[]
lgb_test_preds=[]; xgb_test_preds=[]
oof_preds = np.zeros(len(X))

print('Training 5-Fold CV: LightGBM + XGBoost Ensemble')
print('-'*60)
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[ti], X.iloc[vi]
    y_tr, y_val = y.iloc[ti], y.iloc[vi]

    lm = lgb.LGBMClassifier(**lgb_params)
    lm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
           callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    lv = lm.predict_proba(X_val)[:,1]; lgb_auc = roc_auc_score(y_val, lv)
    lgb_cv_scores.append(lgb_auc); lgb_test_preds.append(lm.predict_proba(X_test)[:,1])

    xm = xgb.XGBClassifier(**xgb_params)
    xm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xv = xm.predict_proba(X_val)[:,1]; xgb_auc = roc_auc_score(y_val, xv)
    xgb_cv_scores.append(xgb_auc); xgb_test_preds.append(xm.predict_proba(X_test)[:,1])

    ens = 0.6*lv + 0.4*xv; ens_auc = roc_auc_score(y_val, ens)
    ens_cv_scores.append(ens_auc); oof_preds[vi] = ens
    print(f'Fold {fold+1}:  LGB={lgb_auc:.5f}  XGB={xgb_auc:.5f}  Ensemble={ens_auc:.5f}')

print('-'*60)
print(f'LGB  Mean: {np.mean(lgb_cv_scores):.5f} +/- {np.std(lgb_cv_scores):.5f}')
print(f'XGB  Mean: {np.mean(xgb_cv_scores):.5f} +/- {np.std(xgb_cv_scores):.5f}')
print(f'Ens  Mean: {np.mean(ens_cv_scores):.5f} +/- {np.std(ens_cv_scores):.5f}')
print(f'OOF  AUC : {roc_auc_score(y, oof_preds):.5f}')

In [ ]:
# CV performance + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(5); w = 0.28
ax = axes[0]
ax.bar(x-w, lgb_cv_scores, width=w, color=GREEN, alpha=0.85, label='LightGBM')
ax.bar(x,   xgb_cv_scores, width=w, color=BLUE,  alpha=0.85, label='XGBoost')
ax.bar(x+w, ens_cv_scores, width=w, color=GOLD,  alpha=0.85, label='Ensemble')
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i+1}' for i in range(5)], fontsize=10)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('Cross-Validation AUC by Fold', fontsize=12, color='#e2e8f0', pad=10, fontweight='600')
ax.set_ylim(0.75, 0.92)
ax.axhline(np.mean(ens_cv_scores), color=GOLD, linestyle='--', alpha=0.5, linewidth=1.5, label=f'Ens Mean ({np.mean(ens_cv_scores):.4f})')
ax.legend(fontsize=9, facecolor='#111827', edgecolor='#1e2d45'); ax.grid(axis='y', alpha=0.3)
ax2 = axes[1]
fpr, tpr, _ = roc_curve(y, oof_preds); auc_val = roc_auc_score(y, oof_preds)
ax2.plot(fpr, tpr, color=GOLD, linewidth=2.5, label=f'OOF ROC (AUC={auc_val:.5f})')
ax2.plot([0,1],[0,1], color='#4b6080', linestyle='--', linewidth=1)
ax2.fill_between(fpr, tpr, alpha=0.08, color=GOLD)
ax2.set_xlabel('False Positive Rate', fontsize=11); ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('Out-of-Fold ROC Curve', fontsize=12, color='#e2e8f0', pad=10, fontweight='600')
ax2.legend(fontsize=10, facecolor='#111827', edgecolor='#1e2d45'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Feature Importance

In [ ]:
fi_df = (pd.DataFrame({'Feature':X.columns,'Importance':lm.feature_importances_})
         .sort_values('Importance', ascending=True).tail(25))
fig, ax = plt.subplots(figsize=(11, 9))
colors = [GOLD if v > fi_df['Importance'].max()*0.5 else '#3b6b9e' for v in fi_df['Importance']]
bars = ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, height=0.7, alpha=0.85)
for bar, val in zip(bars, fi_df['Importance']):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', ha='left', fontsize=9, color='#94a3b8')
ax.set_xlabel('Importance Score (Gain)', fontsize=11)
ax.set_title('Top 25 Feature Importances — LightGBM (Final Fold)', fontsize=13, color='#e2e8f0', pad=14, fontweight='600')
ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()
print('Top 5 features:')
print(fi_df.tail(5)[['Feature','Importance']].iloc[::-1].to_string(index=False))

## 6. Generate Submission File

In [ ]:
lgb_final = np.mean(lgb_test_preds, axis=0)
xgb_final = np.mean(xgb_test_preds, axis=0)
final_preds = 0.6*lgb_final + 0.4*xgb_final
print(f'Prediction range: {final_preds.min():.4f} to {final_preds.max():.4f}')
print(f'Mean prediction : {final_preds.mean():.4f}  (train draft rate: {y.mean():.4f})')

In [ ]:
# Save submission.csv
submission = pd.read_csv(PATH / 'sample_submission.csv')
submission['Drafted'] = final_preds
submission.to_csv(PATH / 'submission.csv', index=False)
print('submission.csv saved successfully.')
print(submission.head(8))

In [ ]:
# Prediction distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(final_preds, bins=50, color=GOLD, alpha=0.85, edgecolor='none')
ax.axvline(0.5,  color=RED,   linestyle='--', linewidth=1.5, label='Threshold (0.5)')
ax.axvline(final_preds.mean(), color=GREEN, linestyle='--', linewidth=1.5, label=f'Mean ({final_preds.mean():.3f})')
ax.set_xlabel('Predicted Draft Probability', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Distribution of Predicted Draft Probabilities (Test Set)', fontsize=12, color='#e2e8f0', pad=10, fontweight='600')
ax.legend(fontsize=9, facecolor='#111827', edgecolor='#1e2d45'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()